In [11]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from collections import Counter


In [12]:
def build_vocab(texts, min_freq=3):
    counter=Counter()
    for text in texts:
        counter.update(text.split())

    vocab = {"<pad>": 0, "<unk>": 1, "<eos>": 2}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    
    return vocab

class NextTokenDataset(Dataset):
    def __init__(self, texts, vocab, max_len=32):
        self.samples = []
        for text in texts:
            ids = [vocab.get(w, 1) for w in text.split()] + [2]
            if len(ids) < 2:
                continue
            ids = ids[:max_len]
            self.samples.append((ids[:-1], ids[1:]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

def collate_fn(batch):
    xs, ys = zip(*batch)
    xs = torch.nn.utils.rnn.pad_sequence(xs, batch_first=True, padding_value=0)
    ys = torch.nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=0)
    return xs, ys


In [16]:
train_texts = pd.read_csv("/home/rays/Загрузки/Projects/text-autocomplete/data/train.csv")["text"].tolist()
val_texts   = pd.read_csv("/home/rays/Загрузки/Projects/text-autocomplete/data/val.csv")["text"].tolist()
test_texts = pd.read_csv("/home/rays/Загрузки/Projects/text-autocomplete/data/test.csv")["text"].tolist()

vocab = build_vocab(train_texts)
print(f"Размер словаря: {len(vocab)}")

train_dataset = NextTokenDataset(train_texts, vocab)
val_dataset   = NextTokenDataset(val_texts,   vocab)
test_dataset = NextTokenDataset(test_texts, vocab)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=256, shuffle=False, collate_fn=collate_fn)

x, y = next(iter(train_loader))
print(f"Batch X: {x.shape}, Batch Y: {y.shape}")



Размер словаря: 111909
Batch X: torch.Size([256, 31]), Batch Y: torch.Size([256, 31])
